# Chinook Database — SQLAlchemy ORM Exercises

## Exercise 1 — Open the Database

In [14]:
import sqlalchemy
import os

print('Current working directory:', os.getcwd())

# Create engine with correct database path
db_path = r'sqlite:///c:\Users\hp\Geeks_Institute\Achraf_Moualem\week4\Day3\chinook\chinook.db'
engine = sqlalchemy.create_engine(db_path)
cur = engine.connect()

print('Connected:', engine)

# Reflect the database and prepare ORM classes
metadata = sqlalchemy.MetaData()
metadata.reflect(engine)

from sqlalchemy.ext.automap import automap_base

Base = automap_base(metadata=metadata)
Base.prepare()

from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)
session = Session()

print('ORM session ready.')

Current working directory: c:\Users\hp\Geeks_Institute\Achraf_Moualem\week4\Day3
Connected: Engine(sqlite:///c:\Users\hp\Geeks_Institute\Achraf_Moualem\week4\Day3\chinook\chinook.db)
ORM session ready.


## Exercise 2 — Table Names

In [17]:
print('Tables in the database:')
for name in metadata.tables.keys():
    print(' -', name)

Tables in the database:
 - albums
 - artists
 - customers
 - employees
 - genres
 - invoice_items
 - tracks
 - media_types
 - invoices
 - playlist_track
 - playlists


## Exercise 3 — First 3 Tracks

In [16]:
Track = Base.classes.tracks

tracks = session.query(Track).limit(3).all()

for t in tracks:
    print(f'ID: {t.TrackId} | Name: {t.Name} | Composer: {t.Composer} | Duration: {t.Milliseconds} ms')

ID: 1 | Name: For Those About To Rock (We Salute You) | Composer: Angus Young, Malcolm Young, Brian Johnson | Duration: 343719 ms
ID: 2 | Name: Balls to the Wall | Composer: None | Duration: 342562 ms
ID: 3 | Name: Fast As a Shark | Composer: F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman | Duration: 230619 ms


## Exercise 4 — Track Name + Album Title (first 20)

In [18]:
Album = Base.classes.albums

results = (
    session.query(Track.Name, Album.Title)
    .join(Album, Track.AlbumId == Album.AlbumId)
    .limit(20)
    .all()
)

print(f'{'Track Name':<45} {'Album Title'}')
print('-' * 75)
for track_name, album_title in results:
    print(f'{track_name:<45} {album_title}')

Track Name                                    Album Title
---------------------------------------------------------------------------
For Those About To Rock (We Salute You)       For Those About To Rock We Salute You
Put The Finger On You                         For Those About To Rock We Salute You
Let's Get It Up                               For Those About To Rock We Salute You
Inject The Venom                              For Those About To Rock We Salute You
Snowballed                                    For Those About To Rock We Salute You
Evil Walks                                    For Those About To Rock We Salute You
C.O.D.                                        For Those About To Rock We Salute You
Breaking The Rules                            For Those About To Rock We Salute You
Night Of The Long Knives                      For Those About To Rock We Salute You
Spellbound                                    For Those About To Rock We Salute You
Balls to the Wall         

## Exercise 5 — Tracks Sold (first 10 sales)

In [23]:
InvoiceItem = Base.classes.invoice_items

# First 10 raw sales
print('--- First 10 invoice items ---')
items = session.query(InvoiceItem).limit(10).all()
for item in items:
    print(f'InvoiceLineId: {item.InvoiceLineId} | TrackId: {item.TrackId} | Quantity: {item.Quantity}')

print()

# Track names + quantity for those 10 sales
print('--- Track names for first 10 sales ---')
results = (
    session.query(Track.Name, InvoiceItem.Quantity)
    .join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId)
    .limit(10)
    .all()
)
print(f'{'Track Name':<45} Quantity')
print('-' * 55)
for name, qty in results:
    print(f'{name:<45} {qty}')

--- First 10 invoice items ---
InvoiceLineId: 1 | TrackId: 2 | Quantity: 1
InvoiceLineId: 2 | TrackId: 4 | Quantity: 1
InvoiceLineId: 3 | TrackId: 6 | Quantity: 1
InvoiceLineId: 4 | TrackId: 8 | Quantity: 1
InvoiceLineId: 5 | TrackId: 10 | Quantity: 1
InvoiceLineId: 6 | TrackId: 12 | Quantity: 1
InvoiceLineId: 7 | TrackId: 16 | Quantity: 1
InvoiceLineId: 8 | TrackId: 20 | Quantity: 1
InvoiceLineId: 9 | TrackId: 24 | Quantity: 1
InvoiceLineId: 10 | TrackId: 28 | Quantity: 1

--- Track names for first 10 sales ---
Track Name                                    Quantity
-------------------------------------------------------
Balls to the Wall                             1
Restless and Wild                             1
Put The Finger On You                         1
Inject The Venom                              1
Evil Walks                                    1
Breaking The Rules                            1
Dog Eat Dog                                   1
Overdose                           

## Exercise 6 — Top 10 Tracks Sold

In [24]:
from sqlalchemy import func

top_tracks = (
    session.query(Track.Name, func.sum(InvoiceItem.Quantity).label('total_sold'))
    .join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId)
    .group_by(Track.TrackId)
    .order_by(func.sum(InvoiceItem.Quantity).desc())
    .limit(10)
    .all()
)

print(f'{'Track Name':<45} Times Sold')
print('-' * 57)
for name, total in top_tracks:
    print(f'{name:<45} {total}')

Track Name                                    Times Sold
---------------------------------------------------------
Balls to the Wall                             2
Inject The Venom                              2
Snowballed                                    2
Overdose                                      2
Deuces Are Wild                               2
Not The Doctor                                2
Por Causa De Você                             2
Welcome Home (Sanitarium)                     2
Snowblind                                     2
Cornucopia                                    2


## Exercise 7 — Top 10 Selling Artists

In [25]:
Artist = Base.classes.artists

top_artists = (
    session.query(Artist.Name, func.sum(InvoiceItem.Quantity).label('total_sold'))
    .join(Album,  Artist.ArtistId == Album.ArtistId)
    .join(Track,  Track.AlbumId   == Album.AlbumId)
    .join(InvoiceItem, InvoiceItem.TrackId == Track.TrackId)
    .group_by(Artist.ArtistId)
    .order_by(func.sum(InvoiceItem.Quantity).desc())
    .limit(10)
    .all()
)

print(f'{'Artist':<35} Total Tracks Sold')
print('-' * 52)
for name, total in top_artists:
    print(f'{name:<35} {total}')

Artist                              Total Tracks Sold
----------------------------------------------------
Iron Maiden                         140
U2                                  107
Metallica                           91
Led Zeppelin                        87
Os Paralamas Do Sucesso             45
Deep Purple                         44
Faith No More                       42
Lost                                41
Eric Clapton                        40
R.E.M.                              39
